<a href="https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/ml-humanities-notebooks/02_Alice_generatore_trigrammi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio extra 2 — Alice e la macchina delle tre parole 🐇
### Un generatore letterario a trigrammi, senza reti neurali

**Obiettivo (15–20 minuti).** Scarichiamo *Alice's Adventures in Wonderland* di **Lewis Carroll** (1865), testo di pubblico dominio, e costruiamo un piccolo modello che impara a continuare una sequenza di **due parole** guardando quali terze parole le seguono nel romanzo.

In altre parole, il modello impara le frequenze dei **trigrammi**: tre token consecutivi. Poi genera un nuovo testo facendo scelte casuali **pesate dalle frequenze**.

**Fonte:** [Project Gutenberg, eBook #11](https://www.gutenberg.org/ebooks/11). Il notebook richiede Internet per scaricare il libro. Un modello a trigrammi può **riprodurre passaggi letterali** del testo: genera per combinazione statistica, non comprende la trama e non garantisce originalità.

▶️ In Google Colab: **Ambiente di esecuzione → Esegui tutte**. Poi cambiate il seme casuale e osservate che cosa succede.

## 1. Leggiamo il romanzo

Useremo solo il **corpo dell'opera**, eliminando intestazioni e note editoriali di Project Gutenberg. Tutte le librerie impiegate fanno già parte di Python, tranne `pandas` usato per una piccola tabella.

In [2]:
import re, random, urllib.request
from collections import defaultdict, Counter
import pandas as pd
from IPython.display import display

# Estratto autentico (capitolo I) incorporato per far funzionare il laboratorio anche offline.
ESTRATTO_ALICE = "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, 'and what is the use of a book,' thought Alice 'without pictures or conversation?'\n\nSo she was considering in her own mind (as well as she could, for the hot day made her feel very sleepy and stupid), whether the pleasure of making a daisy-chain would be worth the trouble of getting up and picking the daisies, when suddenly a White Rabbit with pink eyes ran close by her.\n\nThere was nothing so VERY remarkable in that; nor did Alice think it so VERY much out of the way to hear the Rabbit say to itself, 'Oh dear! Oh dear! I shall be late!' (when she thought it over afterwards, it occurred to her that she ought to have wondered at this, but at the time it all seemed quite natural); but when the Rabbit actually TOOK A WATCH OUT OF ITS WAISTCOAT-POCKET, and looked at it, and then hurried on, Alice started to her feet, for it flashed across her mind that she had never before seen a rabbit with either a waistcoat-pocket, or a watch to take out of it, and burning with curiosity, she ran across the field after it, and fortunately was just in time to see it pop down a large rabbit-hole under the hedge.\n\nIn another moment down went Alice after it, never once considering how in the world she was to get out again.\n\nThe rabbit-hole went straight on like a tunnel for some way, and then dipped suddenly down, so suddenly that Alice had not a moment to think about stopping herself before she found herself falling down a very deep well.\n\nEither the well was very deep, or she fell very slowly, for she had plenty of time as she went down to look about her and to wonder what was going to happen next. First, she tried to look down and make out what she was coming to, but it was too dark to see anything; then she looked at the sides of the well, and noticed that they were filled with cupboards and book-shelves; here and there she saw maps and pictures hung upon pegs. She took down a jar from one of the shelves as she passed; it was labelled 'ORANGE MARMALADE', but to her great disappointment it was empty: she did not like to drop the jar for fear of killing somebody, so managed to put it into one of the cupboards as she fell past it.\n\n'Well!' thought Alice to herself, 'after such a fall as this, I shall think nothing of tumbling down stairs! How brave they'll all think me at home! Why, I wouldn't say anything about it, even if I fell off the top of the house!' (Which was very likely true.)\n\nDown, down, down. Would the fall NEVER come to an end! 'I wonder how many miles I've fallen by this time?' she said aloud. 'I must be getting somewhere near the centre of the earth. Let me see: that would be four thousand miles down, I think--' (for, you see, Alice had learnt several things of this sort in her lessons in the schoolroom, and though this was not a VERY good opportunity for showing off her knowledge, as there was no one to listen to her, still it was good practice to say it over) '--yes, that's about the right distance--but then I wonder what Latitude or Longitude I've got to?' (Alice had no idea what Latitude was, or Longitude either, but thought they were nice grand words to say.)\n\nPresently she began again. 'I wonder if I shall fall right THROUGH the earth! How funny it'll seem to come out among the people that walk with their heads downward! The Antipathies, I think--' (she was rather glad there WAS no one listening, this time, as it didn't sound at all the right word) '--but I shall have to ask them what the name of the country is, you know. Please, Ma'am, is this New Zealand or Australia?' (and she tried to curtsey as she spoke--fancy CURTSEYING as you're falling through the air! Do you think you could manage it?) 'And what an ignorant little girl she'll think me for asking! No, it'll never do to ask: perhaps I shall see it written up somewhere.'\n\nDown, down, down. There was nothing else to do, so Alice soon began talking again. 'Dinah'll miss me very much to-night, I should think!' (Dinah was the cat.) 'I hope they'll remember her saucer of milk at tea-time. Dinah my dear! I wish you were down here with me! There are no mice in the air, I'm afraid, but you might catch a bat, and that's very like a mouse, you know. But do cats eat bats, I wonder?' And here Alice began to get rather sleepy, and went on saying to herself, in a dreamy sort of way, 'Do cats eat bats? Do cats eat bats?' and sometimes, 'Do bats eat cats?' for, you see, as she couldn't answer either question, it didn't much matter which way she put it. She felt that she was dozing off, and had just begun to dream that she was walking hand in hand with Dinah, and saying to her very earnestly, 'Now, Dinah, tell me the truth: did you ever eat a bat?' when suddenly, thump! thump! down she came upon a heap of sticks and dry leaves, and the fall was over.\n\nAlice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, hurrying down it. There was not a moment to be lost: away went Alice like the wind, and was just in time to hear it say, as it turned a corner, 'Oh my ears and whiskers, how late it's getting!' She was close behind it when she turned the corner, but the Rabbit was no longer to be seen: she found herself in a long, low hall, which was lit up by a row of lamps hanging from the roof.\n\nThere were doors all round the hall, but they were all locked; and when Alice had been all the way down one side and up the other, trying every door, she walked sadly down the middle, wondering how she was ever to get out again.\n\nSuddenly she came upon a little three-legged table, all made of solid glass; there was nothing on it except a tiny golden key, and Alice's first thought was that it might belong to one of the doors of the hall; but, alas! either the locks were too large, or the key was too small, but at any rate it would not open any of them. However, on the second time round, she came upon a low curtain she had not noticed before, and behind it was a little door about fifteen inches high: she tried the little golden key in the lock, and to her great delight it fitted!\n\nAlice opened the door and found that it led into a small passage, not much larger than a rat-hole: she knelt down and looked along the passage into the loveliest garden you ever saw. How she longed to get out of that dark hall, and wander about among those beds of bright flowers and those cool fountains, but she could not even get her head through the doorway; 'and even if my head would go through,' thought poor Alice, 'it would be of very little use without my shoulders. Oh, how I wish I could shut up like a telescope! I think I could, if I only know how to begin.' For, you see, so many out-of-the-way things had happened lately, that Alice had begun to think that very few things indeed were really impossible."
URLS = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "https://www.gutenberg.org/files/11/11-0.txt",
    "https://raw.githubusercontent.com/fmasanori/PPZ/master/alice.txt",
]

def solo_romanzo(s):
    # Nei file Gutenberg cerchiamo il segnale di inizio e fine.
    m = re.search(r"\*\*\*\s*START OF (?:(?:THE|THIS) )?PROJECT GUTENBERG EBOOK[^\n]*\*\*\*", s, re.I)
    if m: s = s[m.end():]
    fine = re.search(r"\*\*\*\s*END OF (?:(?:THE|THIS) )?PROJECT GUTENBERG EBOOK[^\n]*\*\*\*", s, re.I)
    if fine: s = s[:fine.start()]
    return s

testo = None
for url in URLS:
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=6) as r:
            candidato = r.read().decode("utf-8-sig", errors="replace")
        candidato = solo_romanzo(candidato)
        if len(candidato) >= 20000:
            testo = candidato
            print("✅ Testo completo caricato da:", url)
            break
    except Exception as err:
        print("Fonte non disponibile:", type(err).__name__)

if testo is None:
    testo = ESTRATTO_ALICE
    print("✅ Uso l'ESTRATTO originale incorporato (nessun download richiesto)")

print("Caratteri:", len(testo))
assert len(testo) > 1000

Fonte non disponibile: HTTPError
Fonte non disponibile: HTTPError
✅ Testo completo caricato da: https://raw.githubusercontent.com/fmasanori/PPZ/master/alice.txt
Caratteri: 147883


## 2. Da un romanzo a una lista di token

La **tokenizzazione** suddivide una stringa in unità. Qui ogni parola inglese è un token; anche `.`, `,`, `!`, `?`, `;`, `:` sono token. Manteniamo le maiuscole: così una generazione può iniziare con `Alice` invece di `alice`.

In [3]:
token = re.findall(r"[A-Za-z]+(?:['’][A-Za-z]+)?|[.,!?;:]", testo)

print("Token totali:", len(token))
print("Primi 30 token:")
print(token[:30])

assert len(token) > 15000, "Il testo sembra incompleto: controlla il download."

Token totali: 31192
Primi 30 token:
["ALICE'S", 'ADVENTURES', 'IN', 'WONDERLAND', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '.', 'CHAPTER', 'I', '.', 'Down', 'the', 'Rabbit', 'Hole', 'Alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister']


## 3. Che cos'è un trigramma?

Se il testo contiene `Alice was beginning to get`, i trigrammi sono:

- `Alice was beginning`
- `was beginning to`
- `beginning to get`

Il modello cerca, ad esempio, tutte le occorrenze di `Alice was` e conta quale parola viene dopo.

La regola statistica è:

\[P(w_3\mid w_1,w_2)=\frac{\text{conteggio}(w_1,w_2,w_3)}{\text{conteggio}(w_1,w_2)}\]

**Non** sta indovinando l'emozione o il significato: sta contando successioni di token.

In [4]:
# Ogni coppia (prima parola, seconda parola) ha un contatore
# delle possibili terze parole.
trigrammi = defaultdict(Counter)

for i in range(len(token) - 2):
    prima = token[i]
    seconda = token[i + 1]
    terza = token[i + 2]
    trigrammi[(prima, seconda)][terza] += 1

print("Coppie di partenza differenti:", len(trigrammi))

# Se 'Alice was' non comparisse nel corpus, scegliamo una coppia frequente.
contesto = ("Alice", "was")
if contesto not in trigrammi:
    contesto = max(trigrammi, key=lambda coppia: sum(trigrammi[coppia].values()))

print("Contesto scelto:", contesto)

Coppie di partenza differenti: 15193
Contesto scelto: ('Alice', 'was')


## 4. Guardiamo le probabilità

La stessa coppia di parole può essere seguita da più alternative. Più spesso compare una continuazione nel romanzo, **più alta** è la probabilità che il nostro generatore la scelga.

In [5]:
conteggi = trigrammi[contesto]
totale = sum(conteggi.values())

possibili = pd.DataFrame(
    [(parola, numero, numero / totale) for parola, numero in conteggi.most_common(12)],
    columns=["prossima parola", "occorrenze", "probabilità"]
)
display(possibili)
print("Somma delle probabilità per il contesto:", round(sum(conteggi.values()) / totale, 3))

,prossima parola,occorrenze,probabilità
0,not,3,0.176471
1,beginning,2,0.117647
2,very,2,0.117647
3,soon,1,0.058824
4,so,1,0.058824
5,more,1,0.058824
6,just,1,0.058824
7,a,1,0.058824
8,silent,1,0.058824
9,rather,1,0.058824


Somma delle probabilità per il contesto: 1.0


## 5. Generiamo un testo nuovo! 🎩

Partiamo dalle due parole del `contesto`. A ogni giro:

1. cerchiamo quali parole potrebbero venire dopo;
2. ne estraiamo una a caso, rispettando i conteggi nel libro;
3. spostiamo in avanti la finestra di due parole.

Il parametro **seme** rende ripetibile l'estrazione casuale: stesso seme → stesso risultato. Cambiatelo per ottenere una nuova continuazione.

In [6]:
def genera(tabelle_trigrammi, coppia_iniziale, n_token=110, seme=7):
    rng = random.Random(seme)
    parole_generate = list(coppia_iniziale)

    for _ in range(n_token - 2):
        coppia = tuple(parole_generate[-2:])
        if coppia not in tabelle_trigrammi:
            break  # non conosciamo una continuazione per questa coppia
        conteggi = tabelle_trigrammi[coppia]
        alternative = list(conteggi.keys())
        frequenze = list(conteggi.values())
        parola_estratta = rng.choices(alternative, weights=frequenze, k=1)[0]
        parole_generate.append(parola_estratta)

    return parole_generate


def ricomponi(parole):
    frase = " ".join(parole)
    return re.sub(r"\s+([.,!?;:])", r"\1", frase)

# Cambiate questi valori per sperimentare!
SEME = 7
LUNGHEZZA = 110

nuovi_token = genera(trigrammi, contesto, n_token=LUNGHEZZA, seme=SEME)
print(ricomponi(nuovi_token))
print(f"\nToken generati: {len(nuovi_token)}")

Alice was soon submitted to by all three to settle the question? said the Gryphon. Of course twinkling begins with a pair of white kid gloves and a large one, time for dinner! I wouldn't say anything. Why, what would happen next. It's a friend of mine a Cheshire cat, said the King. Nearly two miles high, and knocked. There's a porpoise. Wouldn't it really? said Alice, shall I NEVER get any older than you, she thought, till she got to the other. As if I only wish it was: at any

Token generati: 110


## 6. Cambiamo punto di partenza

Scegliamo una coppia tra quelle **realmente osservate** nel romanzo, evitando di inventare un contesto inesistente. Per una coppia più rara, la continuazione è spesso più vincolata (e può copiare lunghe sequenze).

In [7]:
contesti_frequenti = [
    coppia for coppia, successivi in trigrammi.items()
    if sum(successivi.values()) >= 7
    and coppia[0].isalpha()
    and coppia[0][0].isupper()
]

print("Alcuni contesti di partenza disponibili:")
print(contesti_frequenti[:15])

# Potete modificare l'indice nella prossima riga.
altro_contesto = contesti_frequenti[3] if len(contesti_frequenti) >= 4 else contesto
print("\nNuovo contesto:", altro_contesto)
print(ricomponi(genera(trigrammi, altro_contesto, n_token=90, seme=21)))

Alcuni contesti di partenza disponibili:
[('Alice', 'was'), ('So', 'she'), ('White', 'Rabbit'), ('There', 'was'), ('I', 'shall'), ('Alice', 'had'), ('Alice', 'to'), ('Why', ','), ('I', 'wonder'), ('I', 'must'), ('I', 'think'), ('Do', 'you'), ('No', ','), ('I', 'should'), ('I', 'wish')]

Nuovo contesto: ('There', 'was')
There was not an encouraging opening for a moment that it seemed quite natural to Alice: he had taken advantage of the Mock Turtle. And she squeezed herself up closer to Alice's side as she spoke, but, alas for poor Alice began telling them her adventures from the shock of being upset, and oh dear! I shall be punished for it, said the sage, as soon as look at all, said Alice, and every now and then turned to the


## 🎲 Mini-esercizio a gruppi (5 minuti)

1. **Previsione:** dalla coppia `Alice was`, quali parole vi aspettate? Confrontatele con la tabella delle probabilità.
2. **Esperimento:** mettete `SEME = 8`, poi `SEME = 9` nella cella 5; il testo cambia? Quali sequenze rimangono identiche?
3. **Discussione:** trovate un punto in cui la storia sembra sensata e uno in cui si rompe. Il modello conosce i personaggi o solo successioni di parole?
4. **Sfida:** scegliete una coppia meno frequente in `trigrammi` e generate da quella. La continuazione risulta più prevedibile?

**Per andare oltre:** che cosa cambierebbe usando un modello a **2-grammi** (una sola parola di contesto) o a **4-grammi** (tre parole di contesto)? Discutete il compromesso tra varietà, coerenza locale e rischio di copiare l'originale.

💡 **Idea da portare a casa:** una parte dell'apparente fluidità del linguaggio si può ottenere con probabilità locali, anche senza reti neurali e senza comprensione semantica.